In [1]:
# ============================================
# CELLULE 1 - INSTALLATION & IMPORTS
# ============================================

# Installer les librairies nécessaires
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib -q

# Importer les librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score

# XGBoost
import xgboost as xgb

# Statsmodels pour time series
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

print("✅ Toutes les librairies sont importées")
print(f"✅ XGBoost version : {xgb.__version__}")

✅ Toutes les librairies sont importées
✅ XGBoost version : 3.2.0


In [2]:
# ============================================
# CELLULE 2 - UPLOAD DES FICHIERS CSV
# ============================================

from google.colab import files
import io

print("="*60)
print("UPLOAD DES FICHIERS CSV")
print("="*60)
print("""
📤 Veuillez uploader les 10 fichiers suivants :
   1. Dim_Canal_Distribution.csv
   2. Dim_Client.csv
   3. Dim_Commandes.csv
   4. Dim_Date.csv
   5. Dim_Fournisseur.csv
   6. Dim_Matiere_Premiere.csv
   7. Dim_Produits.csv
   8. Dim_Reclamations.csv
   9. Fact_Achat.csv
   10. Fact_Vente.csv

👉 Conseil : Sélectionnez les 10 fichiers en même temps (Ctrl+clic ou Cmd+clic)
""")

# Upload des fichiers
uploaded = files.upload()

print(f"\n✅ {len(uploaded)} fichiers uploadés avec succès !")
print(f"📁 Fichiers : {list(uploaded.keys())}")

# Charger les fichiers dans des dataframes
dfs = {}
for nom_fichier in uploaded.keys():
    nom_base = nom_fichier.replace('.csv', '').replace('.CSV', '')
    dfs[nom_base] = pd.read_csv(io.BytesIO(uploaded[nom_fichier]))
    print(f"   ✅ {nom_base} : {dfs[nom_base].shape[0]} lignes, {dfs[nom_base].shape[1]} colonnes")

# Assigner aux variables
df_canal = dfs.get('Dim_Canal_Distribution')
df_client = dfs.get('Dim_Client')
df_commandes = dfs.get('Dim_Commandes')
df_date = dfs.get('Dim_Date')
df_fournisseur = dfs.get('Dim_Fournisseur')
df_mp = dfs.get('Dim_Matiere_Premiere')
df_produits = dfs.get('Dim_Produits')
df_reclamations = dfs.get('Dim_Reclamations')
df_achat = dfs.get('Fact_Achat')
df_vente = dfs.get('Fact_Vente')

print("\n✅ Tous les fichiers sont chargés et prêts !")

UPLOAD DES FICHIERS CSV

📤 Veuillez uploader les 10 fichiers suivants :
   1. Dim_Canal_Distribution.csv
   2. Dim_Client.csv
   3. Dim_Commandes.csv
   4. Dim_Date.csv
   5. Dim_Fournisseur.csv
   6. Dim_Matiere_Premiere.csv
   7. Dim_Produits.csv
   8. Dim_Reclamations.csv
   9. Fact_Achat.csv
   10. Fact_Vente.csv

👉 Conseil : Sélectionnez les 10 fichiers en même temps (Ctrl+clic ou Cmd+clic)



Saving Dim_Canal_Distribution.csv to Dim_Canal_Distribution.csv
Saving Dim_Client.csv to Dim_Client.csv
Saving Dim_Commandes.csv to Dim_Commandes.csv
Saving Dim_Date.csv to Dim_Date.csv
Saving Dim_Fournisseur.csv to Dim_Fournisseur.csv
Saving Dim_Matiere_Premiere.csv to Dim_Matiere_Premiere.csv
Saving Dim_Produits.csv to Dim_Produits.csv
Saving Dim_Reclamations.csv to Dim_Reclamations.csv
Saving Fact_Achat.csv to Fact_Achat.csv
Saving Fact_Vente.csv to Fact_Vente.csv

✅ 10 fichiers uploadés avec succès !
📁 Fichiers : ['Dim_Canal_Distribution.csv', 'Dim_Client.csv', 'Dim_Commandes.csv', 'Dim_Date.csv', 'Dim_Fournisseur.csv', 'Dim_Matiere_Premiere.csv', 'Dim_Produits.csv', 'Dim_Reclamations.csv', 'Fact_Achat.csv', 'Fact_Vente.csv']
   ✅ Dim_Canal_Distribution : 4 lignes, 6 colonnes
   ✅ Dim_Client : 446 lignes, 10 colonnes
   ✅ Dim_Commandes : 442 lignes, 8 colonnes
   ✅ Dim_Date : 1461 lignes, 12 colonnes
   ✅ Dim_Fournisseur : 67 lignes, 7 colonnes
   ✅ Dim_Matiere_Premiere : 62 lignes

In [5]:
# ============================================
# CELLULE 3 - CRÉATION DU DATASET PRINCIPAL (CORRIGÉE)
# ============================================

print("="*60)
print("CRÉATION DU DATASET PRINCIPAL")
print("="*60)

# Nettoyer les clés dans df_vente
print("\n🔧 Nettoyage des clés dans df_vente :")
for col in ['produit_key', 'client_key', 'date_key', 'commande_key', 'canal_key']:
    if col in df_vente.columns:
        df_vente[col] = df_vente[col].fillna(0).astype(int)
        print(f"   ✅ {col} nettoyé")

# RENOMMER les colonnes dans les tables dimensions pour correspondre
print("\n🔧 Harmonisation des noms de colonnes :")

# Renommer Client_key → client_key
if 'Client_key' in df_client.columns:
    df_client = df_client.rename(columns={'Client_key': 'client_key'})
    print("   ✅ df_client : Client_key → client_key")

# Renommer Produit_key → produit_key
if 'Produit_key' in df_produits.columns:
    df_produits = df_produits.rename(columns={'Produit_key': 'produit_key'})
    print("   ✅ df_produits : Produit_key → produit_key")

# Renommer Commande_key → commande_key
if 'Commande_key' in df_commandes.columns:
    df_commandes = df_commandes.rename(columns={'Commande_key': 'commande_key'})
    print("   ✅ df_commandes : Commande_key → commande_key")

# Vérifier les clés dans df_date (déjà correct : date_key)
if 'date_key' in df_date.columns:
    print("   ✅ df_date : date_key OK")

# Vérifier les clés dans df_canal (déjà correct : canal_key)
if 'canal_key' in df_canal.columns:
    print("   ✅ df_canal : canal_key OK")

# Jointures progressives
df = df_vente.copy()
print(f"\n📊 Dataset initial : {df.shape}")

# Jointure avec clients
if 'client_key' in df.columns and 'client_key' in df_client.columns:
    df = df.merge(df_client, on='client_key', how='left')
    print(f"   ✅ Après jointure clients : {df.shape}")
else:
    print(f"   ⚠️ Jointure clients impossible")

# Jointure avec produits
if 'produit_key' in df.columns and 'produit_key' in df_produits.columns:
    df = df.merge(df_produits, on='produit_key', how='left')
    print(f"   ✅ Après jointure produits : {df.shape}")
else:
    print(f"   ⚠️ Jointure produits impossible")

# Jointure avec dates
if 'date_key' in df.columns and 'date_key' in df_date.columns:
    df = df.merge(df_date, on='date_key', how='left')
    print(f"   ✅ Après jointure dates : {df.shape}")
else:
    print(f"   ⚠️ Jointure dates impossible")

# Jointure avec commandes
if 'commande_key' in df.columns and 'commande_key' in df_commandes.columns:
    df = df.merge(df_commandes, on='commande_key', how='left')
    print(f"   ✅ Après jointure commandes : {df.shape}")
else:
    print(f"   ⚠️ Jointure commandes impossible")

# Jointure avec canaux
if 'canal_key' in df.columns and 'canal_key' in df_canal.columns:
    df = df.merge(df_canal, on='canal_key', how='left')
    print(f"   ✅ Après jointure canaux : {df.shape}")
else:
    print(f"   ⚠️ Jointure canaux impossible")

print(f"\n✅ Dataset final : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")

# Afficher les colonnes disponibles
print(f"\n📋 Colonnes du dataset final :")
print(df.columns.tolist())

CRÉATION DU DATASET PRINCIPAL

🔧 Nettoyage des clés dans df_vente :
   ✅ produit_key nettoyé
   ✅ client_key nettoyé
   ✅ date_key nettoyé
   ✅ commande_key nettoyé
   ✅ canal_key nettoyé

🔧 Harmonisation des noms de colonnes :
   ✅ df_client : Client_key → client_key
   ✅ df_produits : Produit_key → produit_key
   ✅ df_commandes : Commande_key → commande_key
   ✅ df_date : date_key OK
   ✅ df_canal : canal_key OK

📊 Dataset initial : (1386, 14)
   ✅ Après jointure clients : (1386, 23)
   ✅ Après jointure produits : (1386, 32)
   ✅ Après jointure dates : (1386, 43)
   ✅ Après jointure commandes : (1386, 50)
   ✅ Après jointure canaux : (1386, 55)

✅ Dataset final : 1,386 lignes, 55 colonnes

📋 Colonnes du dataset final :
['vente_key', 'date_key', 'client_key', 'canal_key', 'produit_key', 'commande_key', 'reclamation_key', 'quantite', 'prix_unitaire_ht', 'montant_ht', 'montant_tva', 'montant_ttc', 'montant_remise', 'revenue', 'Code_client', 'Nom_client', 'Type_client', 'Ville', 'Gouvern

In [6]:
# ============================================
# CELLULE 3.5 - NETTOYAGE ET CRÉATION DES CIBLES
# ============================================

print("="*60)
print("NETTOYAGE ET CRÉATION DES CIBLES")
print("="*60)

# 1. Supprimer les anomalies
initial_rows = len(df)
df = df[df['produit_key'] != 0]
df = df[df['quantite'] > 0]
df = df[df['montant_ht'] > 0]
print(f"✅ Après suppression anomalies : {len(df):,} lignes (supprimé {initial_rows - len(df):,})")

# 2. Gérer les valeurs manquantes
print("\n📝 Gestion des valeurs manquantes :")
for col in ['Mode_paiement', 'Gouvernorat', 'Ville', 'Sous_Categorie', 'Matiere']:
    if col in df.columns:
        df[col] = df[col].fillna('Non renseigné')
        print(f"   ✅ {col} → 'Non renseigné'")

for col in ['Prix_Promo', 'Frais_livraison', 'Remise_panier', 'montant_remise']:
    if col in df.columns:
        df[col] = df[col].fillna(0)
        print(f"   ✅ {col} → 0")

# 3. Nettoyer les montants avec virgule
def clean_float(x):
    if pd.isna(x):
        return 0.0
    if isinstance(x, str):
        try:
            return float(x.replace(',', '.'))
        except:
            return 0.0
    try:
        return float(x)
    except:
        return 0.0

for col in ['montant_ttc', 'montant_ht', 'montant_tva', 'revenue']:
    if col in df.columns:
        df[col] = df[col].apply(clean_float)
        print(f"   ✅ {col} nettoyé")

# 4. Création des cibles
print("\n🎯 Création des cibles :")

if 'Statut_commande' in df.columns:
    df['est_terminee'] = (df['Statut_commande'] == 'Terminée').astype(int)
    print(f"   ✅ cible classification : est_terminee")
    print(f"      Distribution : {df['est_terminee'].value_counts().to_dict()}")

if 'montant_ttc' in df.columns:
    df['montant_ttc_clean'] = df['montant_ttc'].fillna(df['montant_ht'] * 1.19)
    print(f"   ✅ cible régression : montant_ttc_clean")
    print(f"      Min : {df['montant_ttc_clean'].min():.2f}, Max : {df['montant_ttc_clean'].max():.2f}")

# 5. Convertir les colonnes catégorielles
print("\n📊 Conversion des types :")
categorical_cols = ['Type_client', 'Statut_commande', 'Mode_paiement', 'type_canal',
                    'physique_ou_digital', 'saison', 'Categorie', 'Gouvernorat']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')
        print(f"   ✅ {col} → category")

print(f"\n✅ Dataset prêt : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")

NETTOYAGE ET CRÉATION DES CIBLES
✅ Après suppression anomalies : 371 lignes (supprimé 1,015)

📝 Gestion des valeurs manquantes :
   ✅ Mode_paiement → 'Non renseigné'
   ✅ Gouvernorat → 'Non renseigné'
   ✅ Ville → 'Non renseigné'
   ✅ Sous_Categorie → 'Non renseigné'
   ✅ Matiere → 'Non renseigné'
   ✅ Prix_Promo → 0
   ✅ Frais_livraison → 0
   ✅ Remise_panier → 0
   ✅ montant_remise → 0
   ✅ montant_ttc nettoyé
   ✅ montant_ht nettoyé
   ✅ montant_tva nettoyé
   ✅ revenue nettoyé

🎯 Création des cibles :
   ✅ cible classification : est_terminee
      Distribution : {1: 341, 0: 30}
   ✅ cible régression : montant_ttc_clean
      Min : 1.19, Max : 59500.00

📊 Conversion des types :
   ✅ Type_client → category
   ✅ Statut_commande → category
   ✅ Mode_paiement → category
   ✅ type_canal → category
   ✅ physique_ou_digital → category
   ✅ saison → category
   ✅ Categorie → category
   ✅ Gouvernorat → category

✅ Dataset prêt : 371 lignes, 57 colonnes


In [7]:
# ============================================
# CELLULE 4 - FEATURE ENGINEERING
# ============================================

print("="*60)
print("FEATURE ENGINEERING - DOMAIN-BASED APPROACH")
print("="*60)

# 1. Variables temporelles
if 'annee' in df.columns and 'mois' in df.columns:
    df['mois_annee'] = df['annee'].astype(str) + '-' + df['mois'].astype(str).str.zfill(2)
    print("   ✅ mois_annee créé")

if 'date_complete' in df.columns:
    df['date_complete'] = pd.to_datetime(df['date_complete'], errors='coerce')
    df['jour_semaine'] = df['date_complete'].dt.dayofweek
    df['est_weekend_calcule'] = df['jour_semaine'].isin([5, 6]).astype(int)
    print("   ✅ jour_semaine créé")

# 2. Variables financières
if 'montant_ht' in df.columns and 'quantite' in df.columns:
    df['prix_moyen_unitaire'] = df['montant_ht'] / df['quantite']
    print("   ✅ prix_moyen_unitaire créé")

if 'montant_remise' in df.columns:
    df['taux_remise'] = (df['montant_remise'] / (df['montant_ht'] + df['montant_remise'] + 0.001)).fillna(0)
    df['taux_remise'] = df['taux_remise'].clip(0, 1)
    print("   ✅ taux_remise créé")

if 'Frais_livraison' in df.columns and 'quantite' in df.columns:
    df['frais_livraison_par_produit'] = df['Frais_livraison'] / df['quantite']
    print("   ✅ frais_livraison_par_produit créé")

# 3. Variables de performance produit
if 'Prix_Promo' in df.columns:
    df['est_promo'] = (df['Prix_Promo'] > 0).astype(int)
    print("   ✅ est_promo créé")

# 4. Agrégations par client (RFM-like)
print("\n📊 Création des features RFM clients :")

# Vérifier les colonnes disponibles pour RFM
rfm_cols = []
if 'date_key' in df.columns:
    rfm_cols.append('date_key')
if 'commande_key' in df.columns:
    rfm_cols.append('commande_key')
if 'montant_ht' in df.columns:
    rfm_cols.append('montant_ht')
if 'quantite' in df.columns:
    rfm_cols.append('quantite')

if len(rfm_cols) >= 2:
    client_agg = df.groupby('client_key').agg({
        'date_key': 'max',                    # Récence
        'commande_key': 'nunique',            # Fréquence
        'montant_ht': 'sum',                  # Montant
        'quantite': 'sum'                     # Quantité totale
    }).rename(columns={
        'date_key': 'derniere_commande',
        'commande_key': 'nb_commandes',
        'montant_ht': 'ca_total',
        'quantite': 'qte_totale'
    })

    # Fusionner avec le dataset
    df = df.merge(client_agg, on='client_key', how='left')
    print("   ✅ Features RFM client créées")
    print(f"      - derniere_commande (récence)")
    print(f"      - nb_commandes (fréquence)")
    print(f"      - ca_total (montant)")
    print(f"      - qte_totale")
else:
    print("   ⚠️ Pas assez de colonnes pour RFM")

# 5. Variables saisonnières
if 'est_ramadan' in df.columns:
    df['periode_ramadan'] = df['est_ramadan'].astype(int)
    print("   ✅ periode_ramadan créée")

if 'est_weekend' in df.columns:
    df['est_weekend_int'] = df['est_weekend'].astype(int)
    print("   ✅ est_weekend_int créée")

# 6. Variables de délai (si date_complete disponible)
if 'date_complete' in df.columns:
    df['mois_de_l_annee'] = df['date_complete'].dt.month
    df['trimestre_de_l_annee'] = df['date_complete'].dt.quarter
    print("   ✅ mois_de_l_annee et trimestre_de_l_annee créés")

print(f"\n📊 Nombre total de colonnes après feature engineering : {len(df.columns)}")

FEATURE ENGINEERING - DOMAIN-BASED APPROACH
   ✅ mois_annee créé
   ✅ jour_semaine créé
   ✅ prix_moyen_unitaire créé
   ✅ taux_remise créé
   ✅ frais_livraison_par_produit créé
   ✅ est_promo créé

📊 Création des features RFM clients :
   ✅ Features RFM client créées
      - derniere_commande (récence)
      - nb_commandes (fréquence)
      - ca_total (montant)
      - qte_totale
   ✅ periode_ramadan créée
   ✅ est_weekend_int créée
   ✅ mois_de_l_annee et trimestre_de_l_annee créés

📊 Nombre total de colonnes après feature engineering : 72


In [8]:
# ============================================
# CELLULE 5 - ENCODING & SCALING
# ============================================

print("="*60)
print("ENCODING & SCALING")
print("="*60)

from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Label Encoding pour les variables ordinales
label_encoders = {}
ordinal_cols = ['saison', 'Categorie', 'Type_client']

for col in ordinal_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"   ✅ {col} encodé (LabelEncoder)")

# 2. One-Hot Encoding pour les variables nominales
nominal_cols = ['Mode_paiement', 'type_canal', 'physique_ou_digital', 'Gouvernorat']
nominal_cols = [c for c in nominal_cols if c in df.columns]

if nominal_cols:
    df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
    print(f"   ✅ One-Hot Encoding : {len(nominal_cols)} colonnes transformées")

# 3. Sélectionner les colonnes numériques à scaler
numeric_cols_for_scaling = df.select_dtypes(include=[np.number]).columns.tolist()

# Exclure les clés et cibles
exclude_scaler = ['client_key', 'produit_key', 'date_key', 'commande_key', 'canal_key',
                  'reclamation_key', 'vente_key', 'est_terminee', 'montant_ttc_clean',
                  'montant_ttc', 'revenue']
numeric_cols_for_scaling = [c for c in numeric_cols_for_scaling if c not in exclude_scaler]

print(f"\n📊 Colonnes à scaler : {len(numeric_cols_for_scaling)}")
print(f"   {numeric_cols_for_scaling[:10]}...")

# Appliquer StandardScaler
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[numeric_cols_for_scaling] = scaler.fit_transform(df[numeric_cols_for_scaling])

print("\n✅ StandardScaler appliqué")

# Vérification
print(f"\n📊 Statistiques après scaling :")
print(df_scaled[numeric_cols_for_scaling[:5]].describe().round(2))

# Sauvegarder le scaler
import joblib
try:
    joblib.dump(scaler, 'scaler.pkl')
    joblib.dump(label_encoders, 'label_encoders.pkl')
    print("\n✅ Scaler et encodeurs sauvegardés")
except:
    print("\n⚠️ Sauvegarde non effectuée (pas de dossier Drive)")

ENCODING & SCALING
   ✅ saison encodé (LabelEncoder)
   ✅ Categorie encodé (LabelEncoder)
   ✅ Type_client encodé (LabelEncoder)
   ✅ One-Hot Encoding : 4 colonnes transformées

📊 Colonnes à scaler : 33
   ['quantite', 'prix_unitaire_ht', 'montant_ht', 'montant_tva', 'montant_remise', 'Prix_Vente_HT', 'Prix_Promo', 'jour', 'mois', 'trimestre']...

✅ StandardScaler appliqué

📊 Statistiques après scaling :
       quantite  prix_unitaire_ht  montant_ht  montant_tva  montant_remise
count    371.00            371.00      371.00       371.00          371.00
mean      -0.00             -0.00       -0.00        -0.00            0.00
std        1.00              1.00        1.00         1.00            1.00
min       -0.06             -1.14       -0.08        -0.08           -0.09
25%       -0.06             -0.75       -0.08        -0.08           -0.09
50%       -0.06             -0.40       -0.07        -0.07           -0.09
75%       -0.06              0.22       -0.04        -0.04         

In [9]:
# ============================================
# CELLULE 6 - VÉRIFICATION FINALE
# ============================================

print("="*60)
print("VÉRIFICATION FINALE DU DATASET")
print("="*60)

print(f"\n📊 Dataset stats :")
print(f"   Lignes : {len(df_scaled):,}")
print(f"   Colonnes : {len(df_scaled.columns)}")
print(f"   NaN : {df_scaled.isnull().sum().sum()}")

# Vérifier les cibles
if 'est_terminee' in df_scaled.columns:
    print(f"\n✅ Cible classification : est_terminee")
    print(f"   Distribution :")
    print(f"      Terminées : {df_scaled['est_terminee'].sum():,}")
    print(f"      Non terminées : {(len(df_scaled) - df_scaled['est_terminee'].sum()):,}")
    print(f"      Ratio : {df_scaled['est_terminee'].mean()*100:.1f}% terminées")

if 'montant_ttc_clean' in df_scaled.columns:
    print(f"\n✅ Cible régression : montant_ttc_clean")
    print(f"   Min : {df_scaled['montant_ttc_clean'].min():.2f}")
    print(f"   Max : {df_scaled['montant_ttc_clean'].max():.2f}")
    print(f"   Mean : {df_scaled['montant_ttc_clean'].mean():.2f}")
    print(f"   Median : {df_scaled['montant_ttc_clean'].median():.2f}")

print("\n" + "="*60)
print("✅ DATASET PRÊT POUR LE MODEL UNDERSTANDING")
print("="*60)

VÉRIFICATION FINALE DU DATASET

📊 Dataset stats :
   Lignes : 371
   Colonnes : 156
   NaN : 769

✅ Cible classification : est_terminee
   Distribution :
      Terminées : 341
      Non terminées : 30
      Ratio : 91.9% terminées

✅ Cible régression : montant_ttc_clean
   Min : 1.19
   Max : 59500.00
   Mean : 252.74
   Median : 44.03

✅ DATASET PRÊT POUR LE MODEL UNDERSTANDING


In [10]:
# ============================================
# CELLULE 7 - MODEL UNDERSTANDING : CLASSIFICATION
# Logistic Regression
# ============================================

print("="*60)
print("MODEL UNDERSTANDING - LOGISTIC REGRESSION")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         LOGISTIC REGRESSION                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  La régression logistique modélise la probabilité qu'une observation        │
│  appartienne à une classe (ex: commande terminée ou non).                   │
│  Elle applique une fonction sigmoïde à une combinaison linéaire des         │
│  variables pour obtenir une probabilité entre 0 et 1.                       │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           │
│  • C : inverse de la force de régularisation (1.0 par défaut)              │
│  • penalty : 'l2' (Ridge), 'l1' (Lasso), 'elasticnet', 'none'              │
│  • solver : 'lbfgs' (par défaut), 'liblinear' pour petites données         │
│  • max_iter : 100 par défaut                                                │
│  • class_weight : 'balanced' pour gérer les classes déséquilibrées         │
│                                                                             │
│  📌 HYPOTHÈSES                                                              │
│  -----------                                                               │
│  • Les observations sont indépendantes                                      │
│  • Pas de multicolinéarité forte entre les variables                       │
│  • Relation linéaire entre les log-odds et les prédicteurs                 │
│  • Pas d'outliers extrêmes                                                  │
│                                                                             │
│  📌 LIMITES                                                                │
│  --------                                                                  │
│  • Ne capture pas les relations non-linéaires complexes                    │
│  • Sensible aux outliers                                                   │
│  • Performance limitée sur données très déséquilibrées                     │
│  • Nécessite des features indépendantes                                     │
│                                                                             │
│  📌 JUSTIFICATION POUR CE PROJET                                           │
│  -------------------------                                                 │
│  ✅ Interprétable : coefficients donnent l'impact de chaque variable       │
│  ✅ Baseline solide pour comparer avec des modèles plus complexes          │
│  ✅ Rapide à entraîner sur 1386 lignes                                     │
│  ✅ Gère bien les variables catégorielles encodées                         │
│  ✅ Parfait pour prédire si une commande sera terminée (binaire)           │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Exemple d'utilisation
if 'est_terminee' in df_scaled.columns:
    # Préparer des features simples
    features_base = ['quantite', 'Frais_livraison', 'montant_ht']
    features_base = [f for f in features_base if f in df_scaled.columns]

    if features_base:
        X_sample = df_scaled[features_base].fillna(0)
        y_sample = df_scaled['est_terminee']

        from sklearn.linear_model import LogisticRegression
        lr = LogisticRegression(random_state=42, max_iter=1000)
        lr.fit(X_sample, y_sample)

        print("\n📊 Exemple sur données réelles :")
        print(f"   Features utilisées : {features_base}")
        print(f"   Accuracy : {lr.score(X_sample, y_sample):.4f}")
        print(f"   Coefficients :")
        for feat, coef in zip(features_base, lr.coef_[0]):
            print(f"      {feat}: {coef:.4f}")

MODEL UNDERSTANDING - LOGISTIC REGRESSION

┌─────────────────────────────────────────────────────────────────────────────┐
│                         LOGISTIC REGRESSION                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  La régression logistique modélise la probabilité qu'une observation        │
│  appartienne à une classe (ex: commande terminée ou non).                   │
│  Elle applique une fonction sigmoïde à une combinaison linéaire des         │
│  variables pour obtenir une probabilité entre 0 et 1.                       │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │

In [11]:
# ============================================
# CELLULE 8 - MODEL UNDERSTANDING : CLASSIFICATION
# Random Forest
# ============================================

print("="*60)
print("MODEL UNDERSTANDING - RANDOM FOREST CLASSIFIER")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         RANDOM FOREST CLASSIFIER                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  Random Forest construit plusieurs arbres de décision sur des échantillons  │
│  différents (bagging) et aléatoirement sur les variables.                   │
│  La prédiction finale est le vote majoritaire des arbres.                   │
│  Chaque arbre est faible, mais l'ensemble est fort.                         │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           │
│  • n_estimators : nombre d'arbres (100 par défaut)                         │
│  • max_depth : profondeur maximale (None = illimitée)                      │
│  • min_samples_split : minimum d'échantillons pour diviser (2)             │
│  • min_samples_leaf : minimum d'échantillons par feuille (1)               │
│  • max_features : nombre de variables à considérer ('sqrt')                │
│  • bootstrap : échantillonnage avec remise (True)                          │
│  • class_weight : 'balanced' pour classes déséquilibrées                   │
│                                                                             │
│  📌 HYPOTHÈSES                                                              │
│  -----------                                                               │
│  • Pas d'hypothèse forte sur la distribution des données                   │
│  • Gère bien les interactions non-linéaires                                │
│  • Robuste aux outliers                                                     │
│  • Pas de supposition de linéarité                                          │
│                                                                             │
│  📌 LIMITES                                                                │
│  --------                                                                  │
│  • Moins interprétable qu'un seul arbre                                    │
│  • Peut surapprendre si mal paramétré                                      │
│  • Nécessite plus de mémoire et de temps                                   │
│  • Pas de garantie de performance sur petites données                      │
│  • Black-box par rapport à la régression logistique                        │
│                                                                             │
│  📌 JUSTIFICATION POUR CE PROJET                                           │
│  -------------------------                                                 │
│  ✅ Excellente performance sur données tabulaires                          │
│  ✅ Gère les relations non-linéaires (ex: effet promo × saison)            │
│  ✅ Feature importance intégrée pour interprétation                        │
│  ✅ Robuste aux outliers dans les ventes                                   │
│  ✅ Parfait pour notre classification des commandes (1386 lignes)          │
│  ✅ Gère automatiquement les valeurs manquantes                            │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Exemple d'utilisation
if 'est_terminee' in df_scaled.columns:
    features_base = ['quantite', 'Frais_livraison', 'montant_ht']
    features_base = [f for f in features_base if f in df_scaled.columns]

    if features_base:
        X_sample = df_scaled[features_base].fillna(0)
        y_sample = df_scaled['est_terminee']

        from sklearn.ensemble import RandomForestClassifier
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_sample, y_sample)

        print("\n📊 Exemple sur données réelles :")
        print(f"   Features utilisées : {features_base}")
        print(f"   Accuracy : {rf.score(X_sample, y_sample):.4f}")
        print(f"   Feature importance :")
        for feat, imp in zip(features_base, rf.feature_importances_):
            print(f"      {feat}: {imp:.4f}")

MODEL UNDERSTANDING - RANDOM FOREST CLASSIFIER

┌─────────────────────────────────────────────────────────────────────────────┐
│                         RANDOM FOREST CLASSIFIER                             │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  Random Forest construit plusieurs arbres de décision sur des échantillons  │
│  différents (bagging) et aléatoirement sur les variables.                   │
│  La prédiction finale est le vote majoritaire des arbres.                   │
│  Chaque arbre est faible, mais l'ensemble est fort.                         │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                     

In [12]:
# ============================================
# CELLULE 9 - MODEL UNDERSTANDING : CLASSIFICATION
# XGBoost
# ============================================

print("="*60)
print("MODEL UNDERSTANDING - XGBOOST CLASSIFIER")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                              XGBOOST CLASSIFIER                              │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  XGBoost (eXtreme Gradient Boosting) construit des arbres séquentiellement │
│  où chaque nouvel arbre corrige les erreurs du précédent.                   │
│  C'est une version optimisée et régularisée du Gradient Boosting.          │
│  "Boosted trees" = apprentissage des erreurs.                               │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           │
│  • n_estimators : nombre d'arbres (100 par défaut)                         │
│  • learning_rate : taux d'apprentissage (0.3 par défaut)                   │
│  • max_depth : profondeur maximale (6 par défaut)                          │
│  • subsample : fraction des données pour chaque arbre (1.0)                │
│  • colsample_bytree : fraction des features par arbre (1.0)                │
│  • reg_alpha : régularisation L1 (0)                                       │
│  • reg_lambda : régularisation L2 (1)                                      │
│  • scale_pos_weight : pour déséquilibre des classes                        │
│  • early_stopping_rounds : arrêt précoce                                   │
│                                                                             │
│  📌 HYPOTHÈSES                                                              │
│  -----------                                                               │
│  • Pas d'hypothèse paramétrique forte                                      │
│  • Gère les valeurs manquantes automatiquement                             │
│  • Robuste aux outliers modérés                                            │
│  • Les arbres sont ajoutés séquentiellement                                │
│                                                                             │
│  📌 LIMITES                                                                │
│  --------                                                                  │
│  • Très sensible au paramétrage                                            │
│  • Peut surapprendre si trop d'arbres                                      │
│  • Plus complexe à interpréter                                             │
│  • Consommation mémoire plus élevée                                        │
│  • Nécessite plus de tuning que Random Forest                              │
│                                                                             │
│  📌 JUSTIFICATION POUR CE PROJET                                           │
│  -------------------------                                                 │
│  ✅ Souvent meilleures performances que Random Forest                      │
│  ✅ Gère très bien les données tabulaires                                  │
│  ✅ Feature importance intégrée                                            │
│  ✅ Gère automatiquement les valeurs manquantes                            │
│  ✅ Scale_pos_weight pour déséquilibre des commandes terminées             │
│  ✅ Early stopping pour éviter le surapprentissage                         │
│  ✅ Idéal pour notre classification                                        │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Exemple d'utilisation
if 'est_terminee' in df_scaled.columns:
    features_base = ['quantite', 'Frais_livraison', 'montant_ht']
    features_base = [f for f in features_base if f in df_scaled.columns]

    if features_base:
        X_sample = df_scaled[features_base].fillna(0)
        y_sample = df_scaled['est_terminee']

        import xgboost as xgb
        xgb_clf = xgb.XGBClassifier(n_estimators=100, random_state=42,
                                     use_label_encoder=False, eval_metric='logloss')
        xgb_clf.fit(X_sample, y_sample)

        print("\n📊 Exemple sur données réelles :")
        print(f"   Features utilisées : {features_base}")
        print(f"   Accuracy : {xgb_clf.score(X_sample, y_sample):.4f}")
        print(f"   Feature importance :")
        for feat, imp in zip(features_base, xgb_clf.feature_importances_):
            print(f"      {feat}: {imp:.4f}")

MODEL UNDERSTANDING - XGBOOST CLASSIFIER

┌─────────────────────────────────────────────────────────────────────────────┐
│                              XGBOOST CLASSIFIER                              │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  XGBoost (eXtreme Gradient Boosting) construit des arbres séquentiellement │
│  où chaque nouvel arbre corrige les erreurs du précédent.                   │
│  C'est une version optimisée et régularisée du Gradient Boosting.          │
│  "Boosted trees" = apprentissage des erreurs.                               │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│ 

In [13]:
# ============================================
# CELLULE 10 - MODEL UNDERSTANDING : RÉGRESSION
# Linear Regression
# ============================================

print("="*60)
print("MODEL UNDERSTANDING - LINEAR REGRESSION")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                           LINEAR REGRESSION                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  La régression linéaire trouve la meilleure combinaison linéaire des       │
│  variables pour prédire la cible continue (ex: revenue).                   │
│  Elle minimise la somme des carrés des résidus (MSE).                      │
│  y = β₀ + β₁x₁ + β₂x₂ + ... + ε                                            │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           │
│  • fit_intercept : inclure ou non l'ordonnée à l'origine (True)            │
│  • copy_X : copier ou non les données (True)                               │
│  • positive : contraindre les coefficients à être positifs (False)         │
│                                                                             │
│  📌 HYPOTHÈSES                                                              │
│  -----------                                                               │
│  • Linéarité : relation linéaire entre X et y                             │
│  • Indépendance des résidus                                                │
│  • Homoscédasticité : variance constante des résidus                       │
│  • Normalité des résidus (pour inférence)                                  │
│  • Pas de multicolinéarité parfaite                                        │
│                                                                             │
│  📌 LIMITES                                                                │
│  --------                                                                  │
│  • Très sensible aux outliers                                              │
│  • Ne capture pas les relations non-linéaires                              │
│  • Performance médiocre sur données complexes                              │
│  • Les résidus doivent être normalement distribués                         │
│                                                                             │
│  📌 JUSTIFICATION POUR CE PROJET                                           │
│  -------------------------                                                 │
│  ✅ Baseline simple et interprétable                                       │
│  ✅ Coefficients donnent l'impact de chaque variable sur le revenue        │
│  ✅ Rapide à entraîner                                                     │
│  ✅ Permet de vérifier la linéarité des relations                          │
│  ✅ Pour comparer avec modèles plus complexes                              │
│  ✅ Idéal pour comprendre les tendances linéaires                          │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Exemple d'utilisation
if 'montant_ttc_clean' in df_scaled.columns:
    features_base = ['quantite', 'Frais_livraison']
    features_base = [f for f in features_base if f in df_scaled.columns]

    if features_base:
        X_sample = df_scaled[features_base].fillna(0)
        y_sample = df_scaled['montant_ttc_clean']

        from sklearn.linear_model import LinearRegression
        lr = LinearRegression()
        lr.fit(X_sample, y_sample)

        print("\n📊 Exemple sur données réelles :")
        print(f"   Features utilisées : {features_base}")
        print(f"   R² : {lr.score(X_sample, y_sample):.4f}")
        print(f"   Coefficients :")
        for feat, coef in zip(features_base, lr.coef_):
            print(f"      {feat}: {coef:.4f}")
        print(f"   Intercept : {lr.intercept_:.2f}")

MODEL UNDERSTANDING - LINEAR REGRESSION

┌─────────────────────────────────────────────────────────────────────────────┐
│                           LINEAR REGRESSION                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  La régression linéaire trouve la meilleure combinaison linéaire des       │
│  variables pour prédire la cible continue (ex: revenue).                   │
│  Elle minimise la somme des carrés des résidus (MSE).                      │
│  y = β₀ + β₁x₁ + β₂x₂ + ... + ε                                            │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  --

In [14]:
# ============================================
# CELLULE 11 - MODEL UNDERSTANDING : RÉGRESSION
# Random Forest Regressor
# ============================================

print("="*60)
print("MODEL UNDERSTANDING - RANDOM FOREST REGRESSOR")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                         RANDOM FOREST REGRESSOR                              │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  Random Forest Regressor construit plusieurs arbres de régression.         │
│  La prédiction finale est la moyenne des prédictions de tous les arbres.   │
│  Réduit la variance par rapport à un seul arbre de décision.                │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           │
│  • n_estimators : nombre d'arbres (100)                                    │
│  • max_depth : profondeur maximale                                         │
│  • min_samples_split : minimum d'échantillons pour diviser (2)             │
│  • min_samples_leaf : minimum d'échantillons par feuille (1)               │
│  • max_features : nombre de variables ('sqrt' pour régression)             │
│  • bootstrap : échantillonnage avec remise (True)                          │
│                                                                             │
│  📌 HYPOTHÈSES                                                              │
│  -----------                                                               │
│  • Pas d'hypothèse de linéarité                                            │
│  • Gère les interactions complexes                                         │
│  • Robuste aux outliers                                                    │
│  • Pas de supposition sur la distribution                                  │
│                                                                             │
│  📌 LIMITES                                                                │
│  --------                                                                  │
│  • Ne prédit pas en dehors des valeurs d'entraînement                      │
│  • Moins interprétable qu'une régression linéaire                          │
│  • Peut être lourd sur grandes données                                     │
│  • Pas de formule mathématique simple                                      │
│                                                                             │
│  📌 JUSTIFICATION POUR CE PROJET                                           │
│  -------------------------                                                 │
│  ✅ Excellente performance sur données tabulaires                          │
│  ✅ Gère les relations non-linéaires (prix × saison)                       │
│  ✅ Robuste aux outliers dans les ventes                                   │
│  ✅ Feature importance pour comprendre le CA                                │
│  ✅ Idéal pour prédire le revenue (cible continue)                         │
│  ✅ Pas besoin de normaliser les features                                  │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Exemple d'utilisation
if 'montant_ttc_clean' in df_scaled.columns:
    features_base = ['quantite', 'Frais_livraison']
    features_base = [f for f in features_base if f in df_scaled.columns]

    if features_base:
        X_sample = df_scaled[features_base].fillna(0)
        y_sample = df_scaled['montant_ttc_clean']

        from sklearn.ensemble import RandomForestRegressor
        rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
        rf_reg.fit(X_sample, y_sample)

        print("\n📊 Exemple sur données réelles :")
        print(f"   Features utilisées : {features_base}")
        print(f"   R² : {rf_reg.score(X_sample, y_sample):.4f}")
        print(f"   Feature importance :")
        for feat, imp in zip(features_base, rf_reg.feature_importances_):
            print(f"      {feat}: {imp:.4f}")

MODEL UNDERSTANDING - RANDOM FOREST REGRESSOR

┌─────────────────────────────────────────────────────────────────────────────┐
│                         RANDOM FOREST REGRESSOR                              │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  Random Forest Regressor construit plusieurs arbres de régression.         │
│  La prédiction finale est la moyenne des prédictions de tous les arbres.   │
│  Réduit la variance par rapport à un seul arbre de décision.                │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           

In [15]:
# ============================================
# CELLULE 12 - MODEL UNDERSTANDING : CLUSTERING
# K-Means
# ============================================

print("="*60)
print("MODEL UNDERSTANDING - K-MEANS")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                                  K-MEANS                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  K-Means partitionne les données en K groupes où chaque point appartient   │
│  au cluster dont le centre (centroïde) est le plus proche.                 │
│  Objectif : minimiser la somme des distances intra-cluster (inertie).      │
│  Algorithme : initialisation → assignation → mise à jour → itération.      │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           │
│  • n_clusters : nombre de clusters (K) à déterminer via Elbow              │
│  • init : méthode d'initialisation ('k-means++' recommandé)                │
│  • n_init : nombre d'initialisations différentes (10)                      │
│  • max_iter : nombre maximum d'itérations (300)                            │
│  • tol : tolérance pour convergence (1e-4)                                 │
│  • random_state : pour reproductibilité                                    │
│                                                                             │
│  📌 HYPOTHÈSES                                                              │
│  -----------                                                               │
│  • Les clusters sont sphériques (distance euclidienne)                     │
│  • Les clusters sont de taille similaire                                   │
│  • Les clusters sont séparables linéairement                               │
│  • Données doivent être normalisées                                        │
│                                                                             │
│  📌 LIMITES                                                                │
│  --------                                                                  │
│  • K doit être connu ou trouvé (Elbow, silhouette)                        │
│  • Sensible aux outliers                                                   │
│  • Ne gère pas les clusters de formes non-sphériques                       │
│  • Sensible à l'initialisation                                             │
│  • Ne gère pas les clusters de densités variables                          │
│                                                                             │
│  📌 JUSTIFICATION POUR CE PROJET                                           │
│  -------------------------                                                 │
│  ✅ Simple et rapide pour segmenter les clients (RFM)                      │
│  ✅ Interprétable : profils de clients clairs                              │
│  ✅ Visualisation facile avec PCA                                          │
│  ✅ Parfait pour notre segmentation clients (447 clients)                  │
│  ✅ Elbow method pour trouver le bon K                                     │
│  ✅ Résultats facilement actionnables métier                               │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Exemple : calcul RFM
if 'client_key' in df.columns and 'montant_ht' in df.columns:
    # Calculer RFM
    rfm = df.groupby('client_key').agg({
        'date_key': 'max',
        'commande_key': 'nunique',
        'montant_ht': 'sum'
    }).rename(columns={
        'date_key': 'recence',
        'commande_key': 'frequence',
        'montant_ht': 'montant'
    })

    # Nettoyer
    rfm = rfm.fillna(0)

    # Normaliser
    from sklearn.preprocessing import StandardScaler
    scaler_rfm = StandardScaler()
    rfm_scaled = scaler_rfm.fit_transform(rfm)

    # K-Means
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score

    kmeans = KMeans(n_clusters=3, random_state=42)
    labels = kmeans.fit_predict(rfm_scaled)

    print("\n📊 Exemple sur données RFM :")
    print(f"   Nombre de clients : {len(rfm)}")
    print(f"   Inertie : {kmeans.inertia_:.2f}")
    print(f"   Silhouette score : {silhouette_score(rfm_scaled, labels):.4f}")

    # Profilage des clusters
    rfm['cluster'] = labels
    print(f"\n   Profil des clusters :")
    for i in range(3):
        cluster_data = rfm[rfm['cluster'] == i]
        print(f"   Cluster {i}: {len(cluster_data)} clients")
        print(f"      Recence moyenne: {cluster_data['recence'].mean():.0f}")
        print(f"      Frequence moyenne: {cluster_data['frequence'].mean():.1f}")
        print(f"      Montant moyen: {cluster_data['montant'].mean():.0f}")

MODEL UNDERSTANDING - K-MEANS

┌─────────────────────────────────────────────────────────────────────────────┐
│                                  K-MEANS                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  K-Means partitionne les données en K groupes où chaque point appartient   │
│  au cluster dont le centre (centroïde) est le plus proche.                 │
│  Objectif : minimiser la somme des distances intra-cluster (inertie).      │
│  Algorithme : initialisation → assignation → mise à jour → itération.      │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ------------

In [17]:
# ============================================
# CELLULE 13 - MODEL UNDERSTANDING : TIME SERIES
# ARIMA / SARIMA
# ============================================

print("="*60)
print("MODEL UNDERSTANDING - ARIMA / SARIMA")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                              ARIMA / SARIMA                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  ARIMA (AutoRegressive Integrated Moving Average) modélise une série       │
│  temporelle par :                                                          │
│  • AR (AutoRegressive) : relation avec les valeurs passées                 │
│  • I (Integrated) : différenciation pour stationnarité                     │
│  • MA (Moving Average) : relation avec les erreurs passées                 │
│  SARIMA ajoute la composante Saisonnière (P,D,Q,s).                        │
│                                                                             │
│  📌 PARAMÈTRES CLÉS                                                         │
│  ---------------                                                           │
│  • p : ordre AR (nombre de lags utilisés, 0-5)                            │
│  • d : degré de différenciation (0 ou 1 généralement)                      │
│  • q : ordre MA (taille de la moyenne mobile, 0-5)                         │
│  • P : ordre AR saisonnier                                                 │
│  • D : différenciation saisonnière                                         │
│  • Q : ordre MA saisonnier                                                 │
│  • s : période saisonnière (4 pour trimestriel, 12 pour mensuel)           │
│                                                                             │
│  📌 HYPOTHÈSES                                                              │
│  -----------                                                               │
│  • Stationnarité de la série (ou rendue stationnaire)                      │
│  • Pas de tendance ou saisonnalité résiduelle                              │
│  • Les résidus doivent être du bruit blanc                                │
│  • Série temporelle régulièrement espacée                                  │
│                                                                             │
│  📌 LIMITES                                                                │
│  --------                                                                  │
│  • Ne gère pas les séries longues (>1000 points)                           │
│  • Pas de features exogènes (sauf version SARIMAX)                        │
│  • Nécessite une série stationnaire                                        │
│  • Sensible aux outliers                                                   │
│  • Choix des ordres (p,d,q) non trivial                                    │
│                                                                             │
│  📌 JUSTIFICATION POUR CE PROJET                                           │
│  -------------------------                                                 │
│  ✅ Référence pour la prévision temporelle                                 │
│  ✅ Interprétable : tendance et saisonnalité                               │
│  ✅ Tests de stationnarité (ADF, KPSS) intégrés                           │
│  ✅ SARIMA pour capturer la saisonnalité mensuelle                         │
│  ✅ Diagnostic des résidus pour validation                                 │
│  ✅ Baseline pour comparer avec Prophet et XGBoost TS                      │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

# Exemple : création série temporelle CA mensuel
if 'montant_ttc_clean' in df.columns and 'date_complete' in df.columns:
    # Créer une série temporelle
    df['date_complete'] = pd.to_datetime(df['date_complete'], errors='coerce')
    df['mois_annee'] = df['date_complete'].dt.to_period('M')

    ca_mensuel = df.groupby('mois_annee')['montant_ttc_clean'].sum()

    print("\n📊 Exemple sur données réelles :")
    print(f"   Période : de {ca_mensuel.index.min()} à {ca_mensuel.index.max()}")
    print(f"   Nombre de mois : {len(ca_mensuel)}")
    print(f"   CA mensuel moyen : {ca_mensuel.mean():.0f}")
    print(f"   CA mensuel min : {ca_mensuel.min():.0f}")
    print(f"   CA mensuel max : {ca_mensuel.max():.0f}")

    # Test de stationnarité (ADF)
    from statsmodels.tsa.stattools import adfuller

    result = adfuller(ca_mensuel.dropna())
    print(f"\n   Test ADF : p-value = {result[1]:.4f}")
    if result[1] < 0.05:
        print(f"   → Série stationnaire (p < 0.05)")
    else:
        print(f"   → Série non stationnaire (nécessite différenciation)")

MODEL UNDERSTANDING - ARIMA / SARIMA

┌─────────────────────────────────────────────────────────────────────────────┐
│                              ARIMA / SARIMA                                  │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION                                                               │
│  -----------                                                               │
│  ARIMA (AutoRegressive Integrated Moving Average) modélise une série       │
│  temporelle par :                                                          │
│  • AR (AutoRegressive) : relation avec les valeurs passées                 │
│  • I (Integrated) : différenciation pour stationnarité                     │
│  • MA (Moving Average) : relation avec les erreurs passées                 │
│  SARIMA ajoute la composante Saisonnière (P,D,Q,s).                        │
│        

In [18]:
# ============================================
# CELLULE 14 - RÉCAPITULATIF FINAL
# ============================================

print("="*60)
print("BILAN MODEL UNDERSTANDING")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│  MODÈLES COUVERTS                                                           │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ✅ CLASSIFICATION (3 modèles)                                              │
│     • Logistic Regression                                                   │
│     • Random Forest Classifier                                              │
│     • XGBoost Classifier                                                    │
│                                                                             │
│  ✅ RÉGRESSION (2 modèles)                                                  │
│     • Linear Regression                                                     │
│     • Random Forest Regressor                                               │
│                                                                             │
│  ✅ CLUSTERING (1 modèle)                                                   │
│     • K-Means                                                               │
│                                                                             │
│  ✅ TIME SERIES (1 modèle)                                                  │
│     • ARIMA/SARIMA                                                          │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  POUR CHAQUE MODÈLE, NOUS AVONS COUVERT :                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  📌 INTUITION : Comment fonctionne le modèle                                │
│  📌 PARAMÈTRES : Principaux hyperparamètres                                 │
│  📌 HYPOTHÈSES : Suppositions sous-jacentes                                 │
│  📌 LIMITES : Points faibles à connaître                                    │
│  📌 JUSTIFICATION : Pourquoi ce modèle pour Sougui                          │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

print("\n📊 Dataset utilisé :")
print(f"   Lignes : {len(df_scaled):,}")
print(f"   Colonnes : {len(df_scaled.columns)}")
print(f"   Cible classification : 'est_terminee'")
print(f"   Cible régression : 'montant_ttc_clean'")

print("\n✅ MODEL UNDERSTANDING TERMINÉ !")
print("   Prêt pour la partie ML (entraînement des modèles)")

BILAN MODEL UNDERSTANDING

┌─────────────────────────────────────────────────────────────────────────────┐
│  MODÈLES COUVERTS                                                           │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ✅ CLASSIFICATION (3 modèles)                                              │
│     • Logistic Regression                                                   │
│     • Random Forest Classifier                                              │
│     • XGBoost Classifier                                                    │
│                                                                             │
│  ✅ RÉGRESSION (2 modèles)                                                  │
│     • Linear Regression                                                     │
│     • Random Forest Regressor                                               │
│              

In [19]:
# ============================================
# CELLULE 15 - SAUVEGARDE (OPTIONNELLE)
# ============================================

print("="*60)
print("SAUVEGARDE DES DONNÉES")
print("="*60)

# Sauvegarder les dataframes
try:
    from google.colab import drive
    drive.mount('/content/drive')

    df_scaled.to_csv('/content/drive/MyDrive/Projet_Sougui/data_clean_scaled.csv', index=False)
    df.to_csv('/content/drive/MyDrive/Projet_Sougui/data_clean_non_scaled.csv', index=False)

    print("✅ Données sauvegardées dans Google Drive")
    print("   /content/drive/MyDrive/Projet_Sougui/")
except:
    print("⚠️ Sauvegarde non effectuée (Drive non monté ou dossier inexistant)")
    print("   Les données restent disponibles dans la variable 'df_scaled'")

print("\n🎉 FIN DU NOTEBOOK - MODEL UNDERSTANDING")

SAUVEGARDE DES DONNÉES
⚠️ Sauvegarde non effectuée (Drive non monté ou dossier inexistant)
   Les données restent disponibles dans la variable 'df_scaled'

🎉 FIN DU NOTEBOOK - MODEL UNDERSTANDING
